# 전처리 연습 (Tokenize, Cleansing & Normalization, Stemming & Lemmatization)

1. 데이터셋(corpus)을 찾는다. (만들어진 데이터셋, 크롤링, ...)
2. 전처리<br>
    2-1. 의미가 있는 단어단어로 vocabulary<br>
    2-2. corpus -> 토큰화 + 전처리 -> 문장<br>

1. 데이터셋(corpus)을 찾는다. (만들어진 데이터셋, 크롤링,.•)
- 이어지는 실습에 쓸 수 있을 법한 데이터 찾기 Tip
- 여러 문장으로 이루어진 데이터셋이라면 일단 good
    - 리뷰 등 이어지지 않는 짧은 문장 여러 건 ok
    - 소설 등 이어지는 긴 문장 여러 건 ok
- 대화형 데이터셋도 good
    - QnA로 구성되어 있는 corpus도 좋고
    - 일반적인 대화로 구성되어 있는 corpus도 좋아요~
- 특정 도메인에 대한 정보를 담고 있는 데이터셋도 good
2. 전처리를 해본다.
- 텍스트 자체를 깔끔하게 만드는 것까지만

**강조** :  저작권, 사용 문제가 있는 데이터는 불가능

### 데이터 출처
https://github.com/bab2min/corpus

In [ ]:
import pandas as pd

# 1. 데이터셋 주소 설정 (인터넷에 있는 파일을 직접 가리킵니다)
url = "https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/naver_shopping.txt"

# 2. 데이터 불러오기 
# sep='\t': 데이터가 탭(Tab)으로 구분되어 있다는 뜻입니다.
# names: 데이터의 열(Column) 이름을 우리가 직접 정해줍니다.
df = pd.read_csv(url, sep='\t', names=['rating', 'content'])

# 3. 잘 들어왔는지 확인
print(df.head())

   rating                                            content
0       5                                            배공빠르고 굿
1       2                      택배가 엉망이네용 저희집 밑에층에 말도없이 놔두고가고
2       5  아주좋아요 바지 정말 좋아서2개 더 구매했어요 이가격에 대박입니다. 바느질이 조금 ...
3       2  선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다. 전...
4       5                  민트색상 예뻐요. 옆 손잡이는 거는 용도로도 사용되네요 ㅎㅎ


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   rating   200000 non-null  int64 
 1   content  200000 non-null  object
dtypes: int64(1), object(1)
memory usage: 3.1+ MB


In [ ]:
# 토큰화 + 전처리
from konlpy.tag import Okt

okt = Okt()
token = okt.morphs(df['content'][0])


In [ ]:
token

['배공', '빠르고', '굿']

In [ ]:
def load_ko_stopwords(filepath):
    with open(filepath, 'r', encoding='UTF-8') as f:
        return [line.strip() for line in f]
    
ko_stopwords = load_ko_stopwords('ko_stopwords.txt')

def tokenize(text):
    return [word for word, tag in okt.pos(text, norm=True, stem=True) if tag in ['Noun', 'Verb', 'Adjective'] and word not in ko_stopwords]

In [ ]:
tokenize_list = [tokenize(text) for text in df['content']]

In [ ]:
df['tokenize_list'] = tokenize_list

In [ ]:
# df 저장
df.to_csv('tokenized_data.csv', index=False)

In [ ]:
df = pd.read_csv('tokenized_data.csv')

In [ ]:
from nltk.tokenize import word_tokenize #
import nltk

nltk.download('punkt')
nltk.download('vader_lexicon')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rbgh0\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\rbgh0\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [ ]:
# 감성 분석 시도?
def correlation_analysis(df):
    from nltk.sentiment.vader import SentimentIntensityAnalyzer 
    
    analyzer = SentimentIntensityAnalyzer()
    
    df['sentiment_score'] = df['tokenize_list'].apply(lambda x: analyzer.polarity_scores(' '.join(x))['compound'])
    

In [ ]:
correlation_analysis(df)

In [ ]:
df

,rating,content,tokenize_list,sentiment_score
0,5,배공빠르고 굿,"['배공', '빠르다', '굿']",0.0
1,2,택배가 엉망이네용 저희집 밑에층에 말도없이 놔두고가고,"['택배', '엉망', '용', '집', '밑', '층', '말', '놔두다']",0.0
2,5,아주좋아요 바지 정말 좋아서2개 더 구매했어요 이가격에 대박입니다. 바느질이 조금 ...,"['아주', '좋다', '바지', '정말', '좋다', '개', '더', '구매',...",0.0
3,2,선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다. 전...,"['선물', '용', '받다', '전달', '하다', '하다', '상품', '이다'...",0.0
4,5,민트색상 예뻐요. 옆 손잡이는 거는 용도로도 사용되네요 ㅎㅎ,"['민트', '색상', '예쁘다', '옆', '손잡이', '거', '용', '도로'...",0.0
...,...,...,...,...
199995,2,장마라그런가!!! 달지않아요,"['장마', '런가', '달', '않다']",0.0
199996,5,다이슨 케이스 구매했어요 다이슨 슈퍼소닉 드라이기 케이스 구매했어요가격 괜찮고 배송...,"['다이슨', '케이스', '구매', '하다', '다이슨', '슈퍼소닉', '드라이...",0.0
199997,5,로드샾에서 사는것보다 세배 저렴하네요 ㅜㅜ 자주이용할께요,"['로드샾', '살다', '세배', '저렴하다', '자주', '용하다']",0.0
199998,5,넘이쁘고 쎄련되보이네요~,"['넘다', '이쁘다', '쎄다', '되다', '보이다']",0.0


In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()
analyzer.polarity_scores(df['tokenize_list'][0])

Rating과 Sentiment Score의 상관 계수: nan


c:\Users\rbgh0\miniconda3\envs\pystudy_env\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\rbgh0\miniconda3\envs\pystudy_env\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [ ]:
df

,rating,content,tokenize_list,sentiment_score
0,5,배공빠르고 굿,"['배공', '빠르다', '굿']",0.0
1,2,택배가 엉망이네용 저희집 밑에층에 말도없이 놔두고가고,"['택배', '엉망', '용', '집', '밑', '층', '말', '놔두다']",0.0
2,5,아주좋아요 바지 정말 좋아서2개 더 구매했어요 이가격에 대박입니다. 바느질이 조금 ...,"['아주', '좋다', '바지', '정말', '좋다', '개', '더', '구매',...",0.0
3,2,선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다. 전...,"['선물', '용', '받다', '전달', '하다', '하다', '상품', '이다'...",0.0
4,5,민트색상 예뻐요. 옆 손잡이는 거는 용도로도 사용되네요 ㅎㅎ,"['민트', '색상', '예쁘다', '옆', '손잡이', '거', '용', '도로'...",0.0
...,...,...,...,...
199995,2,장마라그런가!!! 달지않아요,"['장마', '런가', '달', '않다']",0.0
199996,5,다이슨 케이스 구매했어요 다이슨 슈퍼소닉 드라이기 케이스 구매했어요가격 괜찮고 배송...,"['다이슨', '케이스', '구매', '하다', '다이슨', '슈퍼소닉', '드라이...",0.0
199997,5,로드샾에서 사는것보다 세배 저렴하네요 ㅜㅜ 자주이용할께요,"['로드샾', '살다', '세배', '저렴하다', '자주', '용하다']",0.0
199998,5,넘이쁘고 쎄련되보이네요~,"['넘다', '이쁘다', '쎄다', '되다', '보이다']",0.0
